In [ ]:
from langchain.tools import tool
import osmnx as ox

@tool
def query_osm(address : str, tags : dict, dist : int) -> list[dict]:
    """
    Запрашивает объекты из OpenStreetMap в радиусе dist метров от адреса address, которые соответствуют тегам tags.
    Возвращает список словарей, каждый из которых содержит информацию об объекте.
    """
    try:
        features = ox.features_from_address(address, tags=tags, dist=dist).drop(columns=['geometry'])
        results = [
            f[~f.isna()].to_dict()
            for _,f in features.iterrows()
        ]
        return results
    except Exception as e:
        return [{"error": str(e)}]

In [ ]:
from urbanomy.methods.agent import Agent

agent = Agent(system_prompt="Ты эксперт по урбанистике. Используй инструменты для доступа к данным из OSM. Не придумывай информацию, основывайся только на результатах вызова инструментов.", tools=[query_osm], debug=True)

print(agent.invoke("""
Дан следующий адрес:
Санкт-Петербург, Белградская улица 28к1
Есть ли в радиусе 3 километров больница? 
"""))